In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\SHIVANI\OneDrive\Attachments\ChurnGuard AI — Customer Churn Prediction & Retention System\customer_churn.csv")

print("Shape:", df.shape)

Shape: (7043, 21)


In [2]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

print("Missing values:", df.isnull().sum().sum())

Missing values: 0


In [3]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-72 Months"
    ]
)

print(df["TenureGroup"].value_counts())

TenureGroup
49-72 Months    2239
0-12 Months     2186
25-48 Months    1594
13-24 Months    1024
Name: count, dtype: int64


In [4]:
df["AvgMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

print(df[[
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "AvgMonthlySpend"
]].head())

   tenure  MonthlyCharges  TotalCharges  AvgMonthlySpend
0       1           29.85         29.85        29.850000
1      34           56.95       1889.50        55.573529
2       2           53.85        108.15        54.075000
3      45           42.30       1840.75        40.905556
4       2           70.70        151.65        75.825000


In [5]:
service_cols = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["ServiceCount"] = 0

for col in service_cols:
    df["ServiceCount"] += (
        df[col].isin(["Yes", "Yes,"])).astype(int)

In [6]:
support_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport"
]

df["SecuritySupportCount"] = 0

for col in support_cols:
    df["SecuritySupportCount"] += (
        df[col] == "Yes"
    ).astype(int)

print(df["SecuritySupportCount"].value_counts().sort_index())

SecuritySupportCount
0    2793
1    1467
2    1372
3     941
4     470
Name: count, dtype: int64


In [7]:
df["IsNewCustomer"] = (
    df["tenure"] <= 12
).astype(int)

print(df["IsNewCustomer"].value_counts())

IsNewCustomer
0    4857
1    2186
Name: count, dtype: int64


In [8]:
monthly_charge_median = df["MonthlyCharges"].median()

df["HighMonthlyCharge"] = (
    df["MonthlyCharges"] > monthly_charge_median
).astype(int)

print("Median Monthly Charge:", monthly_charge_median)
print(df["HighMonthlyCharge"].value_counts())

Median Monthly Charge: 70.35
HighMonthlyCharge
0    3528
1    3515
Name: count, dtype: int64


In [9]:
df["IsMonthToMonth"] = (
    df["Contract"] == "Month-to-month"
).astype(int)

print(df["IsMonthToMonth"].value_counts())

IsMonthToMonth
1    3875
0    3168
Name: count, dtype: int64


In [10]:
df["IsElectronicCheck"] = (
    df["PaymentMethod"] == "Electronic check"
).astype(int)

print(df["IsElectronicCheck"].value_counts())

IsElectronicCheck
0    4678
1    2365
Name: count, dtype: int64


In [11]:
new_features = [
    "TenureGroup",
    "AvgMonthlySpend",
    "ServiceCount",
    "SecuritySupportCount",
    "IsNewCustomer",
    "HighMonthlyCharge",
    "IsMonthToMonth",
    "IsElectronicCheck"
]

print(df[new_features].head())

    TenureGroup  AvgMonthlySpend  ServiceCount  SecuritySupportCount  \
0   0-12 Months        29.850000             1                     1   
1  25-48 Months        55.573529             3                     2   
2   0-12 Months        54.075000             3                     2   
3  25-48 Months        40.905556             3                     3   
4   0-12 Months        75.825000             1                     0   

   IsNewCustomer  HighMonthlyCharge  IsMonthToMonth  IsElectronicCheck  
0              1                  0               1                  1  
1              0                  0               0                  0  
2              1                  0               1                  0  
3              0                  0               0                  0  
4              1                  1               1                  1  


In [12]:
print(
    df.groupby("IsNewCustomer")["Churn"]
    .value_counts(normalize=True)
    .round(3)
)

IsNewCustomer  Churn
0              No       0.829
               Yes      0.171
1              No       0.526
               Yes      0.474
Name: proportion, dtype: float64


In [13]:
print(
    df.groupby("IsMonthToMonth")["Churn"]
    .value_counts(normalize=True)
    .round(3)
)

IsMonthToMonth  Churn
0               No       0.932
                Yes      0.068
1               No       0.573
                Yes      0.427
Name: proportion, dtype: float64


In [14]:
month_to_month_churn = pd.crosstab(
    df["IsMonthToMonth"],
    df["Churn"],
    normalize="index"
) * 100

print(month_to_month_churn.round(2))

Churn              No    Yes
IsMonthToMonth              
0               93.24   6.76
1               57.29  42.71


In [15]:
new_customer_churn = pd.crosstab(
    df["IsNewCustomer"],
    df["Churn"],
    normalize="index"
) * 100

print(new_customer_churn.round(2))

Churn             No    Yes
IsNewCustomer              
0              82.87  17.13
1              52.56  47.44


In [16]:
df["HighRiskCustomer"] = (
    (df["IsNewCustomer"] == 1) &
    (df["IsMonthToMonth"] == 1)
).astype(int)

print(df["HighRiskCustomer"].value_counts())

HighRiskCustomer
0    5049
1    1994
Name: count, dtype: int64


In [17]:
risk_churn = pd.crosstab(
    df["HighRiskCustomer"],
    df["Churn"],
    normalize="index"
) * 100

print(risk_churn.round(2))

Churn                No    Yes
HighRiskCustomer              
0                 83.26  16.74
1                 48.65  51.35


In [18]:
print("Dataset Shape:", df.shape)

print("\nNew Features:")
print(new_features)

print("\nMissing Values:")
print(df.isnull().sum().sum())

Dataset Shape: (7043, 30)

New Features:
['TenureGroup', 'AvgMonthlySpend', 'ServiceCount', 'SecuritySupportCount', 'IsNewCustomer', 'HighMonthlyCharge', 'IsMonthToMonth', 'IsElectronicCheck']

Missing Values:
0


In [19]:
df.to_csv(
    "../data/processed/churn_feature_engineered.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!
